# Bus passenger count

## 1) Setup

In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "notebooks":
    raise RuntimeError(
        "Run this notebook from the project's 'notebooks/' directory."
    )

PROJECT_ROOT = NOTEBOOK_DIR.parent
SRC = (PROJECT_ROOT / "src").resolve()

# Keep src first in sys.path exactly once.
sys.path = [str(SRC), *[p for p in sys.path if p != str(SRC)]]

# Refresh env vars before any wandb init.
load_dotenv(PROJECT_ROOT / ".env", override=True)

import torch
from ultralytics import YOLO
import wandb

print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU only)")
print("WANDB_ENTITY:", os.environ.get("WANDB_ENTITY"))
print("WANDB_PROJECT:", os.environ.get("WANDB_PROJECT"))
print("YOLO import: ok", YOLO)

wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\lospe\_netrc.


CUDA: True NVIDIA GeForce RTX 3090
WANDB_ENTITY: IA901-2026S1-bus-passenger-count
WANDB_PROJECT: bus-passenger-count
YOLO import: ok <class 'ultralytics.models.yolo.model.YOLO'>


wandb: Currently logged in as: cristianmaza (IA901-2026S1-bus-passenger-count) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 2) Experiment Config

Set model/training params and data/preprocessing selection here.

In [ ]:
import config
import datasets

# Model options: "yolo" (fine-tune) or "yolo_raw" (base model eval)
MODEL_NAME = "yolo"
RUN_TRAINING = True

EXPERIMENT_ID = "yolo-test"

# Pick one or more datasets by name (see `datasets.available()`).
# DATASETS = ["InsideBusView"]
DATASETS = ["InsideBusView", "PassengerDeakin", "PassengerDetectionBus"]

# Dataset stage to download/use: "processed" for modeling, or "interim".
DATASET_STAGE = "processed"

# Force a re-download even if local files are present.
FORCE_DOWNLOAD = False

# Frequent training knobs live here.
TRAIN_OVERRIDES = {
    "epochs": 10,
    "batch": 16,
    "imgsz": 640,
    "workers": 4,
    # "device": 0,
}

# Optional eval-only overrides (useful after kernel restart). Leave empty to
# reuse run_dir / data_yaml produced earlier in this notebook.
EVAL_RUN_DIR = ""
EVAL_DATA_YAML = ""

notes = (
    f"model={MODEL_NAME}; run_training={RUN_TRAINING}; "
    f"datasets={DATASETS}; dataset_stage={DATASET_STAGE}; "
    f"train_overrides={TRAIN_OVERRIDES}"
)

cfg = {
    "experiment_id": EXPERIMENT_ID,
    "notes": notes,
    "yolo_overrides": TRAIN_OVERRIDES,
    "datasets": DATASETS,
    "dataset_stage": DATASET_STAGE,
}

print("model:", MODEL_NAME)
print("run training:", RUN_TRAINING)
print("datasets:", DATASETS)
print("dataset stage:", DATASET_STAGE)
print("available datasets:", datasets.available())
print("train overrides:", TRAIN_OVERRIDES)

model: yolo
run training: True
datasets: ['InsideBusView', 'PassengerDeakin', 'PassengerDetectionBus']
available datasets: ['crowdhuman', 'inside-bus-view', 'passenger-deakin', 'passenger-detection-bus']
train overrides: {'epochs': 10, 'batch': 16, 'imgsz': 640, 'workers': 4}


## 3) Download Dataset

Downloads each name in `DATASETS` from Google Drive zip files into
`data/<DATASET_STAGE>/<name>/` if missing, then writes a YOLO `data.yaml` with
absolute image paths. When more than one dataset is selected, a merged yaml is
written under `data/<DATASET_STAGE>/_combined/<name1>+<name2>+...`.

In [3]:
data_yaml = datasets.prepare(
    DATASETS,
    force_download=FORCE_DOWNLOAD,
    stage=DATASET_STAGE,
)
print("dataset:", data_yaml)

datasets: downloading folder https://drive.google.com/drive/folders/1F-LB5fE6owvsw3vq_zXcblI2M_4g4NyE -> C:\Users\lospe\Documents\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\processed\passenger-deakin


Retrieving folder contents


Retrieving folder 1DkOPIa9on1mXikkoj0CgoMFYxLIAC9p5 test
Retrieving folder 14QlK8J1mm3V0-rkt8Zxehk8IW4OZXrt7 images
Processing file 1SXQdn_wsZemcKhWkCtJJZ5dFwwvEYpO7 003-054_jpg.rf.abYe6B8lbKceMFhl0CJO.jpg
Processing file 150xQ0Pw_s2st8nibTWpUPTcQVThZYCbD 009-047_jpg.rf.qfjzmwbb8JfpgT5IPOjv.jpg
Processing file 1XzD9KaICe4yHDscV2C2bFEOunV9WQ41M 010-039_jpg.rf.BHKOddpaKwnrTxALOuRW.jpg
Processing file 1ku3n_SDh4hSz7AwQ7ZnC1XVwek2Iix7Y 012-023_jpg.rf.pWduE5MOczLlFUj4pb0s.jpg
Processing file 1W6guyuB5JulwaspeabUbKY2E67Zn-B-g 019-054_jpg.rf.hCjsHpSTslst38mKz70L.jpg
Processing file 1RX_wueY4aI4ENNfc3sLN16IJ2xsTK8ks 020-106_jpg.rf.GqZWweHQmSY3vY7MeHRp.jpg
Processing file 1sgo2TaM0Q6jWQ4Jh9vMeSYw8r2v8e0SG 022-060_jpg.rf.CuFOnW6FcKq07WMlq139.jpg
Processing file 1aFtMfEnPmBx40C7Vu94zYi0PcLfKI_19 download (11)_jpg.rf.kuEMLTlV161P3plv4CMH.jpg
Processing file 1UPnzc5rAvXfGUDo6khnPL2Fs_5df3iCB frame20157_jpg.rf.bvWwDlKOqSJuFNZbdy6G.jpg
Processing file 1Hun4e1DdwLIFV8wAUtxJAUobLzipO9ht frame_1410_jpg.

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1SXQdn_wsZemcKhWkCtJJZ5dFwwvEYpO7
To: C:\Users\lospe\Documents\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\processed\passenger-deakin\test\images\003-054_jpg.rf.abYe6B8lbKceMFhl0CJO.jpg
100%|██████████| 80.4k/80.4k [00:00<00:00, 1.75MB/s]
Downloading...
From: https://drive.google.com/uc?id=150xQ0Pw_s2st8nibTWpUPTcQVThZYCbD
To: C:\Users\lospe\Documents\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\processed\passenger-deakin\test\images\009-047_jpg.rf.qfjzmwbb8JfpgT5IPOjv.jpg
100%|██████████| 53.5k/53.5k [00:00<00:00, 720kB/s]
Downloading...
From: https://drive.google.com/uc?id=1XzD9KaICe4yHDscV2C2bFEOunV9WQ41M
To: C:\Users\lospe\Documents\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\processed\passenger-deakin\test\images\010-039_jpg.rf.BHKOddpaKwnrTxALOuRW.jpg
100%|██████████| 81.1k/81.1k [0

FileURLRetrievalError: Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1moHKyB6xr52YfZL09fVUsFFWstFztHYi

but Gdown can't. Please check connections and permissions.

## 4) Train (Optional)

Set `RUN_TRAINING = False` in config to skip training and run eval-only.

In [9]:
import train
from datetime import datetime
from pathlib import Path

if RUN_TRAINING:
    run_dir = train.run(model_name=MODEL_NAME, cfg=cfg, data_yaml=data_yaml)
else:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_name = f"{EXPERIMENT_ID}_evalonly_{ts}"
    run_dir = (Path(config.RUNS_DIR) / run_name).resolve()
    run_dir.mkdir(parents=True, exist_ok=True)
    print("train: skipped (eval-only mode)")

print("run_dir:", run_dir)

train:    run dir -> C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\runs\pipeline-smoke-test_20260513_190253


wandb:    run 'pipeline-smoke-test_20260513_190253' -> https://wandb.ai/cristianmaza/bus-passenger-count/runs/ttlvivau
yolo:     loading base weights -> yolo11m.pt
yolo:     training for 10 epochs on C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\processed\passenger-detection-on-a-bus-qgljh-v1\data.yaml
Ultralytics 8.4.50  Python-3.12.5 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24576MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\processed\passenger-detection-on-a-bus-qgljh-v1\data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end

## 5) Evaluate

Runs evaluation and uploads metrics + sample predictions to wandb.

In [9]:
from pathlib import Path
import eval as eval_mod

resolved_run_dir = Path(EVAL_RUN_DIR).resolve() if EVAL_RUN_DIR else globals().get("run_dir")
resolved_data_yaml = Path(EVAL_DATA_YAML).resolve() if EVAL_DATA_YAML else globals().get("data_yaml")

if resolved_run_dir is None:
    raise RuntimeError(
        "run_dir is missing. Run the Train cell first, or set EVAL_RUN_DIR in config."
    )
if resolved_data_yaml is None:
    raise RuntimeError(
        "data_yaml is missing. Run the Download Dataset cell first, or set EVAL_DATA_YAML in config."
    )

eval_weights = None if RUN_TRAINING else config.MODEL_WEIGHTS[MODEL_NAME]

metrics = eval_mod.run(
    run_dir=resolved_run_dir,
    data_yaml=resolved_data_yaml,
    model_name=MODEL_NAME,
    n_samples=10,
    cfg=cfg,
    weights=eval_weights,
)
metrics

eval:     weights -> C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\runs\pipeline-smoke-test_20260513_190253\yolo\weights\best.pt
yolo:     loading weights for eval -> C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\runs\pipeline-smoke-test_20260513_190253\yolo\weights\best.pt
Ultralytics 8.4.50  Python-3.12.5 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24576MiB)
YOLO11m summary (fused): 126 layers, 20,030,803 parameters, 0 gradients, 67.6 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 2418.71045.8 MB/s, size: 472.7 KB)
val: Scanning C:\Users\lospe\Documents\Unicamp\bus_counter\IA901-2026S1\projetos\bus-passenger-count\data\processed\passenger-detection-on-a-bus-qgljh-v1\test\labels.cache... 17 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 17/17  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.4it/s 1.4s3.9s
        

wandb:    logged 10 images under 'test/predictions'


final/test/fitness,▁
final/test/metrics/mAP50(B),▁
final/test/metrics/mAP50-95(B),▁
final/test/metrics/precision(B),▁
final/test/metrics/recall(B),▁
test/fitness,▁
test/metrics/mAP50(B),▁
test/metrics/mAP50-95(B),▁
test/metrics/precision(B),▁
test/metrics/recall(B),▁
final/test/fitness,0.4892


wandb:    run finished


{'test/metrics/precision(B)': 0.8748774492178799,
 'test/metrics/recall(B)': 0.8685446009389671,
 'test/metrics/mAP50(B)': 0.9359662093203561,
 'test/metrics/mAP50-95(B)': 0.4892042724812895,
 'test/fitness': 0.4892042724812895}